# SportContext: NEW matched-training TF-IDF / Sentence-BERT comparison

این نوت‌بوک جدید را روی GPU نوع L4 اجرا کنید. دو سلول کد دارد؛ هر دو را به‌ترتیب اجرا کنید. نوت‌بوک‌های قدیمی را Run all نکنید.

**Two complete training runs are required. This is not an instant inference-only comparison.** Both arms start from the recovered Model 2 epoch-3 checkpoint, use the same seed (42), train rows, two additional epochs, optimizer and decoding. Only the selector representation changes. Each arm is trained with its own selected contexts. No test scores select epochs, thresholds or the winner for deployment.

**New experiment, not a reproduction of the original paper.** The reconstructed selector uses sentence/48-word segmentation, exact-source echo removal, token-chunked SBERT embeddings, a fixed shared utility threshold, and word/token budgets. Different representations can yield different context lengths; report those lengths. The shared threshold is not independently optimized for either representation. A single-seed result does not establish statistical significance. The new initial checkpoint itself comes from the later run and is not the original Model 2 checkpoint.

Results/logs and selected contexts are saved incrementally under `NMT_Sports_Project/metrics/matched_context_comparison_<timestamp>`. Full trained weights remain on **temporary Colab disk**, because Drive is nearly full. A runtime reset can lose those weights and unfinished training; disconnecting the browser does not guarantee that Colab continues. The original weights are never overwritten or deleted. Rerunning the main cell starts a new experiment, not a resume.

The notebook is syntax-checked locally. Training and GPU execution have **not** been tested in this assistant environment. Use the resulting outputs only after reviewing them. No numerical result is pre-filled.

Documentation: https://www.sbert.net/docs/package_reference/sentence_transformer/model.html and https://huggingface.co/docs/transformers/v4.50.0/model_doc/nllb


In [ ]:
from google.colab import drive
import subprocess, sys

drive.mount('/content/drive', force_remount=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.57.1', 'sentence-transformers==5.1.2',
    'sacrebleu==2.6.0', 'sentencepiece', 'pyarrow', 'scikit-learn'], check=True)
print('Dependencies ready. Run the next cell.')


In [ ]:
from pathlib import Path
import subprocess, sys
script = '"""New matched-training experiment; not a reproduction of the original paper."""\nimport os\nos.environ[\'TOKENIZERS_PARALLELISM\'] = \'false\'\nimport gc, json, math, random, re, shutil, time, hashlib, unicodedata\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport numpy as np\nimport pandas as pd\nimport torch\nimport torch.nn.functional as F\nfrom torch.utils.data import DataLoader\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sentence_transformers import SentenceTransformer\nfrom transformers import (AutoTokenizer, AutoModelForSeq2SeqLM,\n                          DataCollatorForSeq2Seq, get_linear_schedule_with_warmup)\nfrom sacrebleu.metrics import BLEU, CHRF\nimport importlib.metadata as metadata\n\nBASE = Path(\'/content/drive/MyDrive/NMT_Sports_Project\')\nINIT = BASE / \'models/Model2_epoch_extension_NEW_20260923_135303/epoch_3_weights\'\nSEED, EPOCHS, BATCH, ACCUM = 42, 2, 2, 16\nLR, WD, WARMUP, SMOOTH = 1e-5, .01, .10, .10\nALPHA, BETA, GAMMA, TAU = 1., .35, .10, .02\nWORDS, INPUT_MAX, OUTPUT_MAX = 180, 512, 128\nSBERT = \'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2\'\nassert torch.cuda.is_available(), \'GPU required: choose L4, then rerun.\'\nassert torch.cuda.is_bf16_supported(), \'This script requires a BF16-capable GPU such as L4.\'\nassert (INIT / \'model.safetensors\').exists(), f\'Missing initial weights: {INIT}\'\nassert shutil.disk_usage(\'/content\').free > 9 * 1024**3, \'Need 9 GB of local Colab disk (not Drive).\'\nSTAMP = datetime.now(timezone.utc).strftime(\'%Y%m%d_%H%M%S\')\nOUT = BASE / \'metrics\' / (\'matched_context_comparison_\' + STAMP)\nLOCAL = Path(\'/content\') / (\'matched_context_models_\' + STAMP)\nOUT.mkdir(parents=True, exist_ok=False)\nLOCAL.mkdir(parents=True, exist_ok=False)\n\ndef save_json(name, value):\n    dest = OUT / name\n    tmp = dest.with_suffix(dest.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(value, ensure_ascii=False, indent=2), encoding=\'utf-8\')\n    tmp.replace(dest)\n\ndef save_csv(name, frame):\n    dest = OUT / name\n    tmp = dest.with_suffix(\'.csv.tmp\')\n    frame.to_csv(tmp, index=False)\n    tmp.replace(dest)\n\ndef seed_all():\n    random.seed(SEED)\n    np.random.seed(SEED)\n    torch.manual_seed(SEED)\n    torch.cuda.manual_seed_all(SEED)\n    torch.backends.cudnn.benchmark = False\n    torch.backends.cudnn.deterministic = True\n\ndef norm(value):\n    s = unicodedata.normalize(\'NFKC\', str(value)).translate(str.maketrans({\'ي\':\'ی\',\'ك\':\'ک\'}))\n    s = re.sub(r\'[\\u200b-\\u200f\\ufeff\\-‐‑–—]\', \' \', s)\n    return re.sub(r\'\\s+\', \' \', s).strip()\n\ndef key(s):\n    return re.sub(r\'\\s+\', \' \', re.sub(r\'[^\\w\\s]\', \' \', norm(s).lower())).strip()\n\nframes = {s: pd.read_parquet(BASE / \'data/splits\' / (s + \'.parquet\'))\n          for s in [\'train\', \'validation\', \'test\']}\nfor s, n, groups in [(\'train\',1023,21), (\'validation\',150,3), (\'test\',150,3)]:\n    f = frames[s]\n    assert len(f) == n and f.context_group_id.nunique() == groups, (s, f.shape)\n    assert f.sample_id.is_unique and not f[[\'source\',\'target\']].isna().any().any()\n    assert f.source.map(key).ne(\'\').all()\naudit = {}\nfor a,b in [(\'train\',\'validation\'),(\'train\',\'test\'),(\'validation\',\'test\')]:\n    counts = {}\n    for c in [\'source\',\'target\']:\n        counts[c] = len(set(frames[a][c].map(key)) & set(frames[b][c].map(key)))\n    counts[\'context_group_id\'] = len(set(frames[a].context_group_id) & set(frames[b].context_group_id))\n    assert not any(counts.values()), (a, b, counts)\n    audit[a + \'/\' + b] = counts\nsave_json(\'split_audit.json\', audit)\n\nprotocol = dict(experiment=\'NEW matched-training representation comparison\',\n    original_paper_reproduction=False, initial_checkpoint=str(INIT), seed=SEED,\n    additional_epochs=EPOCHS, additional_train_rows=1023, augmentation_in_additional_training=False,\n    batch=BATCH, gradient_accumulation=ACCUM, learning_rate=LR, weight_decay=WD,\n    warmup_ratio=WARMUP, label_smoothing=SMOOTH, optimizer=\'torch AdamW\',\n    checkpoint_rule=\'Fixed final epoch 2, no selection or early stopping\',\n    gate=dict(alpha=ALPHA,beta=BETA,gamma=GAMMA,tau=TAU,word_budget=WORDS),\n    input_max_tokens=INPUT_MAX, output_max_length=OUTPUT_MAX, num_beams=4,\n    representation_model=SBERT, tfidf=dict(analyzer=\'char_wb\',ngram_range=[3,5],\n        min_df=2,max_features=50000,sublinear_tf=True),\n    echo_rule=\'Drop candidate containing whole normalized source before ranking; paraphrases not removed\',\n    segmentation=\'Sentence punctuation and segment_sep, then 48-word blocks\',\n    gate_rule=\'Recomputed greedy marginal utility, nonnegative max cosine redundancy; tie by candidate index\',\n    packing=\'source <CTX> context; 180 words and actual NLLB 512-token budget during selection\',\n    embedding_overflow=\'Token chunks with token-count-weighted mean of normalized chunk embeddings\',\n    input_normalization=\'NFKC, Persian character normalization, hyphens and zero-width chars to spaces\',\n    references=\'Original English strings, never decoded training labels\',\n    test_rule=\'Both final fixed-epoch systems scored; never choose or tune using test\',\n    limitation=\'Single seed, reconstructed selector; shared threshold, not independently optimized\',\n    GPU=torch.cuda.get_device_name(0), local_models=str(LOCAL), output=str(OUT),\n    versions={p:metadata.version(p) for p in [\'torch\',\'transformers\',\'sentence-transformers\',\'sacrebleu\']})\nprotocol[\'split_sha256\'] = {s: hashlib.sha256((BASE/\'data/splits\'/(s+\'.parquet\')).read_bytes()).hexdigest() for s in frames}\nsave_json(\'LOCKED_PROTOCOL.json\', protocol)\nprint(\'PROTOCOL LOCKED. Outputs:\', OUT, flush=True)\n\nseed_all()\ntok = AutoTokenizer.from_pretrained(str(INIT), src_lang=\'pes_Arab\', tgt_lang=\'eng_Latn\', local_files_only=True)\ntok.add_special_tokens({\'additional_special_tokens\':[\'<CTX>\']}, replace_additional_special_tokens=False)\neng = tok.convert_tokens_to_ids(\'eng_Latn\')\nassert eng != tok.unk_token_id\n\ndef candidates(row):\n    seen, result = set(), []\n    source_key = key(row.source)\n    for column in [\'context_1_text\',\'context_2_text\',\'context_3_text\']:\n        value = row[column]\n        if pd.isna(value):\n            continue\n        for sentence in re.split(r\'<segment_sep>|[.!?؟؛\\n\\r]+\', str(value)):\n            sentence = norm(sentence)\n            # Remove echoes before splitting; otherwise an echo could straddle blocks.\n            if (\' \' + source_key + \' \') in (\' \' + key(sentence) + \' \'):\n                continue\n            words = sentence.split()\n            for start in range(0, len(words), 48):\n                block = \' \'.join(words[start:start+48])\n                k = key(block)\n                if k and k not in seen:\n                    result.append(block)\n                    seen.add(k)\n    return result\n\nrecords = {}\nfor s,f in frames.items():\n    records[s] = [dict(sample_id=str(r.sample_id),source=norm(r.source),\n                       candidates=candidates(r)) for _,r in f.iterrows()]\n\ntrain_texts = list(dict.fromkeys(t for r in records[\'train\'] for t in [r[\'source\']]+r[\'candidates\']))\nall_texts = list(dict.fromkeys(t for rs in records.values() for r in rs for t in [r[\'source\']]+r[\'candidates\']))\nindex = {t:i for i,t in enumerate(all_texts)}\ntfidf = TfidfVectorizer(analyzer=\'char_wb\',ngram_range=(3,5),min_df=2,\n    max_features=50000,sublinear_tf=True,dtype=np.float32)\ntfidf.fit(train_texts)\nX = tfidf.transform(all_texts)\nprint(\'TF-IDF fitted only on train. Preparing Sentence-BERT.\', flush=True)\n\nencoder = SentenceTransformer(SBERT, device=\'cuda\', cache_folder=\'/content/sbert_cache\')\nencoder.eval()\netok = encoder.tokenizer\nchunk_limit = encoder.max_seq_length - etok.num_special_tokens_to_add(pair=False)\nchunks, owners, lengths = [], [], []\nfor i,t in enumerate(all_texts):\n    ids = etok(t, add_special_tokens=False, truncation=False)[\'input_ids\']\n    for start in range(0, max(1,len(ids)), chunk_limit):\n        piece = ids[start:start+chunk_limit]\n        chunks.append(etok.prepare_for_model(piece, add_special_tokens=True,\n                      return_token_type_ids=False, return_attention_mask=True))\n        owners.append(i)\n        lengths.append(max(1,len(piece)))\nE = np.zeros((len(all_texts), encoder.get_sentence_embedding_dimension()), dtype=np.float32)\nwith torch.inference_mode():\n    for start in range(0,len(chunks),32):\n        batch = etok.pad(chunks[start:start+32],padding=True,return_tensors=\'pt\')\n        assert batch[\'input_ids\'].shape[1] <= encoder.max_seq_length\n        batch = {k:v.to(\'cuda\') for k,v in batch.items()}\n        emb = F.normalize(encoder(batch)[\'sentence_embedding\'],dim=-1).cpu().numpy()\n        for j,v in enumerate(emb):\n            E[owners[start+j]] += lengths[start+j] * v\nE /= np.maximum(np.linalg.norm(E,axis=1,keepdims=True),1e-12)\nsave_json(\'representation_details.json\',dict(tfidf_features=len(tfidf.vocabulary_),\n    sbert_max_seq_length=encoder.max_seq_length, unique_texts=len(all_texts),chunks=len(chunks),\n    model_revision=getattr(encoder[0].auto_model.config,\'_commit_hash\',None)))\ndel encoder, batch, emb\ngc.collect(); torch.cuda.empty_cache()\n\ndef select(r, method):\n    c = r[\'candidates\']\n    source = r[\'source\']\n    assert len(tok(source)[\'input_ids\']) <= INPUT_MAX, \'Source exceeds budget.\'\n    if not c: return \'\'\n    ids = [index[t] for t in c]\n    if method == \'tfidf\':\n        v = X[ids]\n        rel = (v @ X[index[source]].T).toarray().ravel()\n        sim = (v @ v.T).toarray()\n    else:\n        v = E[ids]\n        rel = v @ E[index[source]]\n        sim = v @ v.T\n    remaining, chosen = list(range(len(c))), []\n    wc = np.array([len(t.split()) for t in c])\n    words = 0\n    while remaining:\n        feasible = []\n        for j in remaining:\n            if words + wc[j] > WORDS: continue\n            trial = source + \' <CTX> \' + \' \'.join(c[k] for k in sorted(chosen+[j]))\n            if len(tok(trial)[\'input_ids\']) > INPUT_MAX: continue\n            red = max(0., float(sim[j,chosen].max())) if chosen else 0.\n            u = ALPHA*float(rel[j]) - BETA*red - GAMMA*wc[j]/WORDS\n            feasible.append((u,-j,j))\n        if not feasible: break\n        u,_,j = max(feasible)\n        if u <= TAU: break\n        chosen.append(j); remaining.remove(j); words += wc[j]\n    return \' \'.join(c[j] for j in sorted(chosen))\n\ninputs = {}\nselection_rows = []\nfor method in [\'tfidf\',\'sbert\']:\n    inputs[method] = {}\n    for s,rs in records.items():\n        texts = []\n        for r in rs:\n            ctx = select(r,method)\n            packed = r[\'source\'] + (\' <CTX> \' + ctx if ctx else \'\')\n            texts.append(packed)\n            selection_rows.append(dict(method=method,split=s,sample_id=r[\'sample_id\'],\n                context=ctx,context_words=len(ctx.split()),input_tokens=len(tok(packed)[\'input_ids\'])))\n        inputs[method][s] = texts\n    print(method, \'selection complete\', flush=True)\nsave_csv(\'selected_contexts.csv\',pd.DataFrame(selection_rows))\nsave_csv(\'selection_summary.csv\',pd.DataFrame(selection_rows).groupby([\'method\',\'split\']).agg(\n    rows=(\'sample_id\',\'size\'),mean_context_words=(\'context_words\',\'mean\'),\n    mean_input_tokens=(\'input_tokens\',\'mean\'),empty_contexts=(\'context_words\',lambda x:int((x==0).sum()))).reset_index())\n\nlabels = tok(text_target=frames[\'train\'].target.astype(str).tolist(), truncation=False)[\'input_ids\']\nassert max(map(len,labels)) <= OUTPUT_MAX, \'Target longer than locked output budget.\'\ndef train_data(method):\n    tokens = tok(inputs[method][\'train\'],truncation=False)\n    assert max(map(len,tokens[\'input_ids\'])) <= INPUT_MAX\n    return [dict(input_ids=tokens[\'input_ids\'][i],attention_mask=tokens[\'attention_mask\'][i],\n                 labels=labels[i]) for i in range(len(labels))]\n\nmetric_rows = []\ndef evaluate(model, method, split):\n    model.eval()\n    preds = []\n    t0 = time.perf_counter()\n    for start in range(0,len(inputs[method][split]),4):\n        b = tok(inputs[method][split][start:start+4],padding=True,return_tensors=\'pt\',truncation=False)\n        assert b.input_ids.shape[1] <= INPUT_MAX\n        with torch.inference_mode(), torch.autocast(\'cuda\',dtype=torch.bfloat16):\n            generated = model.generate(**b.to(\'cuda\'),forced_bos_token_id=eng,\n                num_beams=4,max_length=OUTPUT_MAX,do_sample=False,use_cache=True)\n        preds.extend(tok.batch_decode(generated,skip_special_tokens=True))\n    refs = frames[split].target.astype(str).tolist()\n    bleu,chrf = BLEU(),CHRF(word_order=2)\n    row = dict(system=method,split=split,seed=SEED,epochs=EPOCHS,\n        BLEU=bleu.corpus_score(preds,[refs]).score,chrFpp=chrf.corpus_score(preds,[refs]).score,\n        bleu_signature=str(bleu.get_signature()),chrf_signature=str(chrf.get_signature()),\n        generation_seconds=time.perf_counter()-t0, rows=len(preds))\n    out = frames[split][[\'sample_id\',\'source\',\'target\']].copy()\n    out[\'model_input\'] = inputs[method][split]\n    out[\'prediction\'] = preds\n    save_csv(method+\'_\'+split+\'_predictions.csv\',out)\n    metric_rows.append(row)\n    save_csv(\'metrics.csv\',pd.DataFrame(metric_rows))\n    print(json.dumps(row,indent=2),flush=True)\n\nfor method in [\'tfidf\',\'sbert\']:\n    seed_all()\n    print(\'\\nTRAINING:\',method,\'fixed 2 epochs\',flush=True)\n    model = AutoModelForSeq2SeqLM.from_pretrained(str(INIT),local_files_only=True,\n                torch_dtype=torch.float32,attn_implementation=\'eager\')\n    model.resize_token_embeddings(len(tok),mean_resizing=False)\n    model.config.use_cache = False\n    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={\'use_reentrant\':False})\n    model.to(\'cuda\')\n    data = train_data(method)\n    collator = DataCollatorForSeq2Seq(tok,model=model,label_pad_token_id=-100,pad_to_multiple_of=8)\n    optimizer = torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD)\n    batches_per_epoch = math.ceil(len(data)/BATCH)\n    steps_per_epoch = math.ceil(batches_per_epoch/ACCUM)\n    total_steps = EPOCHS*steps_per_epoch\n    scheduler = get_linear_schedule_with_warmup(optimizer,math.ceil(WARMUP*total_steps),total_steps)\n    logs, step = [], 0\n    t0 = time.perf_counter()\n    for epoch in range(EPOCHS):\n        # Same row order for the two arms, independent of dropout RNG consumption.\n        order = np.random.default_rng(SEED+epoch).permutation(len(data))\n        loader = DataLoader([data[int(i)] for i in order],batch_size=BATCH,shuffle=False,\n                            collate_fn=collator,num_workers=0)\n        model.train(); optimizer.zero_grad(set_to_none=True)\n        running = 0.\n        for bi,b in enumerate(loader):\n            b = {k:v.to(\'cuda\') for k,v in b.items()}\n            lab = b.pop(\'labels\')\n            assert \'decoder_input_ids\' in b, \'Decoder inputs not prepared.\'\n            group_start = (bi//ACCUM)*ACCUM\n            group_size = min(ACCUM,len(loader)-group_start)\n            with torch.autocast(\'cuda\',dtype=torch.bfloat16):\n                result = model(**b,use_cache=False)\n                loss = F.cross_entropy(result.logits.reshape(-1,result.logits.shape[-1]),\n                    lab.reshape(-1),ignore_index=-100,label_smoothing=SMOOTH)\n            if not torch.isfinite(loss): raise RuntimeError(\'Nonfinite training loss; stop.\')\n            value = float(loss.detach())\n            (loss/group_size).backward()\n            running += value/group_size\n            del result,loss,b,lab\n            if (bi+1)%ACCUM == 0 or bi+1 == len(loader):\n                torch.nn.utils.clip_grad_norm_(model.parameters(),1.)\n                optimizer.step(); scheduler.step(); optimizer.zero_grad(set_to_none=True)\n                step += 1\n                logs.append(dict(epoch=epoch+1,step=step,loss=running,\n                                 learning_rate=scheduler.get_last_lr()[0]))\n                save_csv(method+\'_training_log.csv\',pd.DataFrame(logs))\n                print(f\'{method} epoch {epoch+1}/2 | step {step}/{total_steps} | loss {running:.4f}\',flush=True)\n                running = 0.\n        save_json(method+\'_status.json\',dict(completed_epochs=epoch+1,\n            seconds=time.perf_counter()-t0,weights_persisted_to_drive=False))\n    del optimizer,scheduler,loader,collator\n    gc.collect(); torch.cuda.empty_cache()\n    model.gradient_checkpointing_disable()\n    model.config.use_cache = True\n    model.save_pretrained(LOCAL/method,safe_serialization=True)\n    tok.save_pretrained(LOCAL/method)\n    evaluate(model,method,\'validation\')\n    del model\n    gc.collect(); torch.cuda.empty_cache()\n\n# Protocol and both trained systems are fixed before any test scoring.\nsave_json(\'BEFORE_TEST_LOCK.json\',dict(methods=[\'tfidf\',\'sbert\'],epochs=EPOCHS,\n    model_paths={m:str(LOCAL/m) for m in [\'tfidf\',\'sbert\']},test_used_for_selection=False))\nfor method in [\'tfidf\',\'sbert\']:\n    model = AutoModelForSeq2SeqLM.from_pretrained(LOCAL/method,local_files_only=True,\n              torch_dtype=torch.float32,attn_implementation=\'eager\').to(\'cuda\')\n    evaluate(model,method,\'test\')\n    del model\n    gc.collect(); torch.cuda.empty_cache()\nsave_json(\'FINISHED.json\',dict(completed=True,output=str(OUT),local_weights=str(LOCAL),\n    interpretation=\'New single-seed matched-training comparison; do not overwrite original paper results.\'))\narchive = shutil.make_archive(\'/content/RESULTS_\'+STAMP,\'zip\',OUT)\nprint(\'\\nALL FINISHED. Results ZIP:\',archive,flush=True)\nprint(pd.DataFrame(metric_rows)[[\'system\',\'split\',\'BLEU\',\'chrFpp\']].to_string(index=False),flush=True)\nprint(\'Keep runtime connected if you need the new model weights; they are on temporary Colab disk.\',flush=True)\n'
p = Path('/content/run_matched_context_comparison.py')
p.write_text(script, encoding='utf-8')
# Fresh Python process avoids stale imports after installation.
subprocess.run([sys.executable, '-u', str(p)], check=True)
